# Moustaghfir Ahmed Yassine - RAG avec PDF

Notebook basé sur la vidéo `j-C9vyyXwTw` : indexing, chunks,
embeddings, base vectorielle, retriever, génération et LLM juge.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

texte_demo = '''
OCP est un groupe marocain spécialisé dans les phosphates et les engrais.
En 2023, le groupe a poursuivi ses investissements industriels et environnementaux.
Le RAG permet de répondre uniquement à partir d'un contexte documentaire fourni.
Si la réponse n'est pas dans le contexte, le modèle doit répondre : je ne sais pas.
'''

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="o200k_base",
    chunk_size=80,
    chunk_overlap=15,
)

chunks = [Document(page_content=chunk) for chunk in splitter.split_text(texte_demo)]
print("Nombre de chunks :", len(chunks))
chunks[0]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class SimpleRetriever:
    def __init__(self, documents, k=3):
        self.documents = documents
        self.k = k
        self.vectorizer = TfidfVectorizer()
        self.matrix = self.vectorizer.fit_transform([doc.page_content for doc in documents])

    def invoke(self, query):
        q = self.vectorizer.transform([query])
        scores = cosine_similarity(q, self.matrix).ravel()
        indices = scores.argsort()[::-1][: self.k]
        return [self.documents[i] for i in indices]

retriever = SimpleRetriever(chunks, k=2)
retriever.invoke("Qu'est-ce que le RAG ?")


In [ ]:
PROMPT_TEMPLATE = '''
Réponds à la question suivante en te basant uniquement sur le contexte fourni.
Si la réponse n'existe pas dans le contexte, réponds : je ne sais pas.

<contexte>
{context}
</contexte>

<question>
{query}
</question>
'''.strip()

def build_context(documents):
    return "\n---\n".join(doc.page_content for doc in documents)

def local_answer(query, context):
    query_lower = query.lower()
    if "rag" in query_lower and "rag" in context.lower():
        return "Le RAG consiste à retrouver un contexte documentaire pertinent puis à générer une réponse à partir de ce contexte."
    if "ocp" in query_lower and "ocp" in context.lower():
        return "OCP est un groupe marocain spécialisé dans les phosphates et les engrais."
    return "je ne sais pas"

def rag(query, llm=None, prompt_template=PROMPT_TEMPLATE):
    docs = retriever.invoke(query)
    context = build_context(docs)
    prompt = prompt_template.format(context=context, query=query)
    if llm is None:
        return local_answer(query, context), context, prompt
    response = llm.invoke(prompt)
    return response.content, context, prompt

answer, context, prompt = rag("Qu'est-ce que le RAG ?")
print(answer)


In [ ]:
question = "Quelle est la capitale de la France ?"
answer, context, prompt = rag(question)
print("Question :", question)
print("Réponse :", answer)


## Version LangChain + Chroma + OpenAI Embeddings (optionnelle)


In [ ]:
# À exécuter si OPENAI_API_KEY est disponible.
# from langchain_community.document_loaders import PyPDFLoader
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from langchain_community.vectorstores import Chroma
#
# loader = PyPDFLoader("PDFs/rapport.pdf")
# docs = loader.load_and_split(splitter)
# vector_store = Chroma.from_documents(
#     documents=docs,
#     embedding=OpenAIEmbeddings(),
#     collection_name="rag_documents",
#     persist_directory="./store",
# )
# retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## Évaluation par LLM juge


In [ ]:
JUDGE_TEMPLATE = '''
Vous êtes chargé d'évaluer une réponse générée par IA.
Question : {question}
Contexte : {context}
Réponse : {answer}

Donnez un score de 1 à 5 :
1 = faux ou hors contexte
3 = partiellement correct
5 = entièrement correct et basé sur le contexte
'''.strip()

def local_judge(question, context, answer):
    if answer == "je ne sais pas":
        return {"score": 3, "reason": "Le modèle refuse de répondre hors contexte."}
    if any(word in context.lower() for word in answer.lower().split()[:5]):
        return {"score": 5, "reason": "La réponse est cohérente avec le contexte."}
    return {"score": 2, "reason": "La réponse semble peu reliée au contexte."}

question = "Qu'est-ce que le RAG ?"
answer, context, _ = rag(question)
local_judge(question, context, answer)
